# Lab Work - 9.2

## Q.1 Distance Metrics Deep Dive

**01 Minkowski Distance**

$$ L_p = \left( \sum |p_i - q_i|^p \right)^{1/p} $$

p=1: Manhattan, p=2: Euclidean, p→∞: Chebyshev

In [ ]:
import numpy as np

def minkowski(a, b, p):
    return np.power(np.sum(np.power(np.abs(a - b), p)), 1/p)

A = np.array([0., 0.])
B = np.array([3., 4.])

print('Manhattan (p=1):', minkowski(A, B, 1))
print('Euclidean (p=2):', minkowski(A, B, 2))
print('Chebyshev:', np.max(np.abs(A - B)))

**02–06 Other Distance Questions (Manhattan, Cosine, Hamming, Curse of Dim, Table)**

In [ ]:
# Cosine
def cosine(a, b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

print('Cosine example:', cosine(np.array([1,0,1]), np.array([1,1,0])))

# Hamming
print('Hamming kitten-sitting:', sum(c1!=c2 for c1,c2 in zip('kitten','sitting')))

## Q.2 Weighted KNN + Scaling (Wine Dataset)

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score, f1_score

wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

for w in ['uniform','distance']:
    knn = KNeighborsClassifier(n_neighbors=7, weights=w)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    print(f'Weights={w}: Acc={accuracy_score(y_test, pred):.4f}')

## Q.3 Hyperparameter Tuning (Breast Cancer)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

param_grid = {'n_neighbors': [3,5,7,9,11], 'weights':['uniform','distance'], 'metric':['euclidean','manhattan']}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='f1_weighted')
grid.fit(X_train_s, y_train)

print('Best params:', grid.best_params_)
print('Best CV score:', grid.best_score_)

## Q.4 Deep Intuition

**KNN intuition:** KNN makes predictions by looking at the labels of the nearest training points and using those neighbors as the “vote.” It is non-parametric, so it does not learn coefficients; it simply stores the training examples and uses distance comparisons at prediction time.

- `k` controls smoothness vs. flexibility. Small `k` makes the model flexible and sensitive to noise, while large `k` makes it smoother and more biased.
- `weights='uniform'` treats each neighbor equally, while `weights='distance'` gives closer neighbors more influence. Weighted voting can improve accuracy when nearby points are more informative than farther ones.
- Distance metric matters. `euclidean` assumes differences are meaningful in raw space, while `manhattan` can be more robust for high-dimensional or sparse features.
- Scaling is essential because KNN uses distance directly. Features with larger numeric ranges dominate the distance calculation, so standardization or normalization ensures each feature contributes fairly.

**Why scaling changes KNN:** without scaling, a feature with range [0,1000] can dominate a feature with range [0,1], effectively making the smaller feature irrelevant. StandardScaler or MinMaxScaler rescales the features so the distance metric reflects meaningful similarity instead of raw magnitude.

**Deeper behavior:** KNN is sensitive to the shape of the data in feature space. In low dimensions, distances preserve nearest neighbors well. As dimensionality increases, distances become less discriminative and the difference between nearest and farthest neighbors shrinks, which is why KNN can degrade in high dimensions.

**Practical takeaways:**
- choose `k` by cross-validation to balance underfitting and overfitting.
- use weighted neighbors when local points vary in importance.
- always scale features before using KNN.
- prefer simpler distance metrics like Euclidean or Manhattan depending on the data distribution and feature scales.

These ideas explain why Q2 and Q3 focused on scaling and grid search: KNN is easy to understand, but its performance depends strongly on distance definition, feature scaling, and the choice of `k`/weighting.